In [1]:
%pip install --upgrade torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --upgrade torch torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

%pip install sentencepiece sacremoses

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

%pip install pandas transformers torch scikit-learn tf-keras

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from sklearn.metrics import accuracy_score

FASE 1

In [6]:
# 1. Leer los archivos
datos_ingles = pd.read_csv("movie_reviews_eng.csv")
datos_frances = pd.read_csv("movie_reviews_fr.csv")
datos_espanol = pd.read_csv("movie_reviews_sp.csv")

In [7]:
# 2. Renombrar las columnas para estandarizar
datos_frances = datos_frances.rename(columns={'Titre': 'Title', 'Année': 'Year', 'Synopsis': 'Synopsis', 'Critiques': 'Review'})
datos_espanol = datos_espanol.rename(columns={'Título': 'Title', 'Año': 'Year', 'Sinopsis': 'Synopsis', 'Críticas': 'Review'})

In [8]:
# 3. Agregar la columna del idioma original
datos_ingles['Original_Language'] = 'English'
datos_frances['Original_Language'] = 'French'
datos_espanol['Original_Language'] = 'Spanish'

In [9]:
# 4. Unificar todo en una sola tabla de 30 filas
tabla_consolidada = pd.concat([datos_ingles, datos_frances, datos_espanol], ignore_index=True)


In [10]:
# Verificamos que tenemos 30 filas
print("Total de filas:", len(tabla_consolidada))

Total de filas: 30


FASE 2

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

c:\Users\Patricia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# Modelos para Francés a Inglés
tokenizador_fr = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
modelo_fr = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-fr-en", use_safetensors=True)

Loading weights: 100%|██████████| 256/256 [00:00<00:00, 3028.44it/s]


In [13]:
# Modelos para Español a Inglés
tokenizador_es = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-es-en")
modelo_es = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-es-en", use_safetensors=True)

Loading weights: 100%|██████████| 256/256 [00:00<00:00, 4279.00it/s]


In [14]:
# Función de traducción 
def traducir_texto_directo(texto, idioma):
    # Convertimos a texto por seguridad para evitar errores con valores nulos
    texto_seguro = str(texto)
    
    if idioma == 'French':
        # Convertir texto a números
        entradas = tokenizador_fr(texto_seguro, return_tensors="pt", padding=True, truncation=True, max_length=512)
        # Generar la traducción matemática
        salidas = modelo_fr.generate(**entradas)
        # Convertir los números de vuelta a texto legible en inglés
        return tokenizador_fr.decode(salidas[0], skip_special_tokens=True)
        
    elif idioma == 'Spanish':
        entradas = tokenizador_es(texto_seguro, return_tensors="pt", padding=True, truncation=True, max_length=512)
        salidas = modelo_es.generate(**entradas)
        return tokenizador_es.decode(salidas[0], skip_special_tokens=True)
        
    else:
        # Si el idioma original ya es inglés, se devuelve intacto
        return texto_seguro

In [15]:
# Aplicamos la función a nuestra tabla consolidada para traducir sinopsis y reseñas
print("Traduciendo sinopsis...")
tabla_consolidada['Synopsis'] = tabla_consolidada.apply(lambda fila: traducir_texto_directo(fila['Synopsis'], fila['Original_Language']), axis=1)

print("Traduciendo reseñas...")
tabla_consolidada['Review'] = tabla_consolidada.apply(lambda fila: traducir_texto_directo(fila['Review'], fila['Original_Language']), axis=1)

print("¡Traducción completada con éxito!")

Traduciendo sinopsis...
Traduciendo reseñas...
¡Traducción completada con éxito!


Fase 3

In [16]:
# ANÁLISIS DE SENTIMIENTO
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch


In [17]:
# 1. Tokenizador y modelo para análisis de sentimiento
nombre_modelo_sentimiento = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizador_sent = AutoTokenizer.from_pretrained(nombre_modelo_sentimiento)
modelo_sent = AutoModelForSequenceClassification.from_pretrained(nombre_modelo_sentimiento, use_safetensors=True)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5193.38it/s]


In [18]:
# Función para clasificar
def obtener_sentimiento_directo(texto):
    texto_seguro = str(texto)
    
    # Convertir a números con límite de longitud para evitar errores
    entradas = tokenizador_sent(texto_seguro, return_tensors="pt", truncation=True, max_length=512)
    
    # torch.no_grad() ahorra memoria y hace el proceso más rápido al no calcular gradientes, ya que solo queremos la predicción
    with torch.no_grad():
        salidas = modelo_sent(**entradas)
    
    # El modelo devuelve probabilidades, Sacamos la más alta.
    # 0 significa NEGATIVE y 1 significa POSITIVE
    prediccion = torch.argmax(salidas.logits, dim=1).item()
    
    if prediccion == 1:
        return "POSITIVE"
    else:
        return "NEGATIVE"

In [19]:
# Aplicamos la función solo a la columna de reseñas para obtener el sentimiento de cada una
print("Analizando los sentimientos de las reseñas...")
tabla_consolidada['Sentiment'] = tabla_consolidada['Review'].apply(obtener_sentimiento_directo)

print("¡Análisis de sentimiento completado con éxito!")

Analizando los sentimientos de las reseñas...
¡Análisis de sentimiento completado con éxito!


Fase 4

In [20]:
import sqlite3

In [21]:
# Filtrar y ordenar las columnas 
columnas_finales = ['Title', 'Year', 'Synopsis', 'Review', 'Original_Language', 'Sentiment']
tabla_final = tabla_consolidada[columnas_finales]

In [22]:
# Exportar a CSV
nombre_archivo = "reviews_with_sentiment.csv"
tabla_final.to_csv(nombre_archivo, index=False)
print(f"Archivo '{nombre_archivo}' generado correctamente.")

Archivo 'reviews_with_sentiment.csv' generado correctamente.


In [23]:
# Guardar en la base de datos SQLite
conexion = sqlite3.connect("base_de_datos_proyecto.db")

In [24]:
# Volcamos el dataframe como una tabla SQL
tabla_final.to_sql("Resenas_Clasificadas", conexion, if_exists="replace", index=False)
conexion.close()

print("Datos cargados exitosamente en la base de datos SQLite.")

Datos cargados exitosamente en la base de datos SQLite.


In [25]:
from sklearn.metrics import accuracy_score, classification_report

In [26]:
# CÁLCULO DEL RENDIMIENTO DEL MODELO

# Define las etiquetas reales manuales para cada reseña
etiquetas_reales_manuales = [
    'POSITIVE', 'NEGATIVE', 'POSITIVE', 'POSITIVE', 'NEGATIVE',
    'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE',
    'POSITIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE',
    'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE', 'POSITIVE',
    'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE',
    'POSITIVE', 'NEGATIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE'
]

# Extrae las predicciones que generó nuestro modelo
predicciones_modelo = tabla_final['Sentiment'].tolist()

# Calcula la exactitud general del modelo comparando las etiquetas reales con las predicciones
exactitud = accuracy_score(etiquetas_reales_manuales, predicciones_modelo)
print(f"\nLa exactitud general del modelo es: {exactitud * 100:.2f}%")

# Genera un reporte detallado que incluye precisión, recall y F1-score para cada clase positiva y negativa
print("\n--- Reporte de Clasificación Detallado ---")
reporte = classification_report(etiquetas_reales_manuales, predicciones_modelo)
print(reporte)


La exactitud general del modelo es: 53.33%

--- Reporte de Clasificación Detallado ---
              precision    recall  f1-score   support

    NEGATIVE       0.53      0.53      0.53        15
    POSITIVE       0.53      0.53      0.53        15

    accuracy                           0.53        30
   macro avg       0.53      0.53      0.53        30
weighted avg       0.53      0.53      0.53        30



In [28]:
# VISUALIZACIÓN DEL RESULTADO FINAL

pd.set_option('display.max_colwidth', None)

# Mostramos las primeras 5 filas para verificar el esquema
print("Vista previa del Esquema de Salida Esperado:")
display(tabla_final)

# Verificación de la estructura completa
print(f"\nDimensiones de la tabla: {tabla_final.shape}")
print(f"Columnas presentes: {list(tabla_final.columns)}")

Vista previa del Esquema de Salida Esperado:


,Title,Year,Synopsis,Review,Original_Language,Sentiment
0,NaN,1994,"Andy Dufresne (Tim Robbins), a successful banker, is wrongfully convicted of murdering his wife and her lover. He befriends a fellow inmate, Red (Morgan Freeman), and uses his banking skills to help the prison staff, eventually gaining their respect.","""The Shawshank Redemption is an inspiring tale of hope and friendship, with outstanding performances by Tim Robbins and Morgan Freeman. The story is incredibly moving and will stay with you long after the credits roll.""",English,POSITIVE
1,NaN,2008,"Batman (Christian Bale) teams up with District Attorney Harvey Dent (Aaron Eckhart) and Police Lieutenant Jim Gordon (Gary Oldman) to take down the Joker (Heath Ledger), who is terrorizing Gotham City.","""The Dark Knight is a thrilling and intense superhero movie, with outstanding performances by Heath Ledger as the Joker and Christian Bale as Batman. The film keeps you on the edge of your seat from start to finish, and the action sequences are some of the best in the genre.""",English,POSITIVE
2,NaN,1994,Forrest Gump (Tom Hanks) is a simple man with a low IQ who becomes involved in some of the major events of the 20th century. He falls in love with his childhood friend Jenny (Robin Wright) and tries to help her through her struggles.,"""Forrest Gump is a heartwarming and inspirational movie, with a fantastic performance by Tom Hanks in the lead role. The film is a powerful reminder of the value of friendship and the importance of perseverance in the face of adversity.""",English,POSITIVE
3,NaN,1972,"Don Vito Corleone (Marlon Brando) is the head of a powerful Mafia family in New York City. When he is shot and nearly killed, his son Michael (Al Pacino) takes over the family business, becoming embroiled in a web of violence and betrayal.","""The Godfather is a classic movie that stands the test of time, with outstanding performances by Marlon Brando and Al Pacino. The film is a masterclass in storytelling, with a gripping plot and memorable characters that keep you engaged from start to finish.""",English,POSITIVE
4,NaN,2010,Dom Cobb (Leonardo DiCaprio) is a skilled thief who enters people's dreams to steal their secrets. He is hired by a wealthy businessman to perform a seemingly impossible task: to implant an idea in someone's mind.,"""Inception is a mind-bending and visually stunning movie, with outstanding performances by Leonardo DiCaprio and the rest of the cast. The film is a masterpiece of storytelling, with a complex plot that keeps you guessing until the very end.""",English,POSITIVE
5,NaN,2017,"Officer K (Ryan Gosling), a new blade runner for the LAPD, unearths a long-buried secret that has the potential to plunge what's left of society into chaos. His discovery leads him on a quest to find Rick Deckard (Harrison Ford), a former blade runner who's been missing for 30 years.","""Boring and too long. Nothing like the original movie."" - IMDb user review",English,NEGATIVE
6,NaN,2010,Scott Pilgrim (Michael Cera) must defeat his new girlfriend's seven evil exes in order to win her heart.,"""It was difficult to sit through the whole thing. Annoying, unfunny and not very interesting."" - Rotten Tomatoes user review",English,NEGATIVE
7,NaN,2016,"In 1970s Los Angeles, a private eye (Ryan Gosling) and a tough enforcer (Russell Crowe) investigate the disappearance of a young woman and the death of a star.","""The Nice Guys tries too hard to be funny, and the result is a disjointed mess of a movie."" - Metacritic user review",English,NEGATIVE
8,NaN,2018,A young Han Solo (Alden Ehrenreich) joins a group of smugglers to pull off a dangerous heist.,"""Dull and pointless, with none of the magic of the original Star Wars movies."" - IMDb user review",English,NEGATIVE
9,NaN,2005,"In a future where people are cloned for organ harvesting, Lincoln Six Echo (Ewan McGregor) and Jordan Two Delta (Scarlett Johansson) escape their facility and go on the run.","""The Island is a bl


Dimensiones de la tabla: (30, 6)
Columnas presentes: ['Title', 'Year', 'Synopsis', 'Review', 'Original_Language', 'Sentiment']
